# Common Statistical Tests as Linear Models

This notebook demonstrates how common statistical tests can be expressed as linear models, following the ideas from Lindeløv (2019).

## Introduction

Many classical statistical tests such as Student’s t-test, Mann-Whitney U test, Wilcoxon signed-rank test, and Wald test can be understood as special cases of linear models.

The key idea:
All these tests evaluate whether a parameter (effect) is significantly different from zero.

This notebook demonstrates this relationship using simulated data and Python.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from scipy.stats import rankdata

np.random.seed(42)

## Generate Example Data

In [ ]:
# Two independent groups
group_A = np.random.normal(5, 1, 30)
group_B = np.random.normal(6, 1, 30)

y = np.concatenate([group_A, group_B])
X = np.array([0]*30 + [1]*30)

X_design = sm.add_constant(X)

## Student’s t-test vs Linear Model

In [ ]:
# Classical t-test
t_stat, p_val = stats.ttest_ind(group_A, group_B)
print("T-test p-value:", p_val)

# Linear model
model = sm.OLS(y, X_design).fit()
print(model.summary())

Interpretation:
- The coefficient of the group variable represents the difference in means.
- The p-value matches the t-test result.

Thus, the t-test is a linear model.

## Mann-Whitney U Test as Linear Model

In [ ]:
# Mann-Whitney
u_stat, p_val = stats.mannwhitneyu(group_A, group_B)
print("Mann-Whitney p-value:", p_val)

# Rank transformation
y_ranked = rankdata(y)

model_rank = sm.OLS(y_ranked, X_design).fit()
print(model_rank.summary())

Interpretation:
- Instead of raw values, we use ranks.
- The regression still tests whether group membership predicts higher/lower ranks.

Thus, Mann-Whitney = linear model on ranks.

## Wilcoxon Signed-Rank Test

In [ ]:
# Paired data
before = np.random.normal(5, 1, 30)
after = before + np.random.normal(0.5, 0.5, 30)

# Wilcoxon test
w_stat, p_val = stats.wilcoxon(before, after)
print("Wilcoxon p-value:", p_val)

# Linear model on differences
diff = after - before
X_diff = sm.add_constant(np.ones(len(diff)))

model_diff = sm.OLS(diff, X_diff).fit()
print(model_diff.summary())

Interpretation:
- We test if the mean difference is zero.
- This is equivalent to testing the intercept in a regression.

Thus, Wilcoxon ≈ linear model on differences.

## Wald Test

In [ ]:
wald_test = model.t_test([0, 1])
print(wald_test)

Interpretation:
- The Wald test evaluates if a coefficient differs from zero.
- This is the core idea behind most statistical tests.

Thus, many tests are simply Wald tests on linear models.

## Conclusion

- Student’s t-test = linear regression with a dummy variable
- Mann-Whitney = regression on ranks
- Wilcoxon = regression on paired differences
- Wald test = general framework for testing coefficients

Overall, many classical statistical tests are simply special cases of linear models.

## Questions / Comments

- How robust is this equivalence under non-normal data?
- Are there cases where classical tests outperform regression approaches?
- How does this extend to generalized linear models?